# Session 1 — Hello, AI

**AI/LLM Application Builder: From Zero to Mastery** · Grow with Data
Instructor: Md Kalim Amzad Chy · Session 1 of 20

Most people meet AI through a chat box. That is the smallest window onto it. By the end of tonight
you will have driven a modern model directly from Python — reading a customer's photo, answering
from the live web with sources, generating an image, and returning strict JSON your code can act on —
and you will know what each of those calls cost.

Over the twenty sessions you build **ShopWise**, a support assistant for a fictional online store
whose inbox is drowning. Tonight is the foundation: the Python this course needs, and a first honest
look at what these models can and cannot do.

---

### How to read this notebook

Everything is written so you can re-learn the session from this file alone, weeks later, with no
recording. **The prose is the lesson; the code is the evidence.** Where something is a trap, the
notebook says so rather than letting you discover it at 1am.

Run cells top to bottom. Outputs are already committed, so the whole notebook is readable without a
key and without spending any quota — but you will learn more by re-running it, because some answers
change between runs and *noticing that* is itself part of the lesson.

A few cells are marked **💳 instructor demo**. Those use a Google capability that needs a
billing-enabled key. They are written so that on a free key they print an explanation instead of
crashing, and the committed output shows you what they return. Nothing in this course *requires* you
to enable billing.

## 1. One cell of housekeeping

This notebook draws diagrams with **mermaid** — a tiny text language for graphs. Rather than ship
picture files that drift out of date, the diagram source lives in the notebook and is rendered when
the cell runs.

`mermaid-py` is already in `labs/pyproject.toml`, so `uv sync` installed it. The `%pip install` line
below is there for anyone running this notebook somewhere else (Google Colab, a bare venv). `%pip`
is a *magic command* — it installs into the kernel actually running this notebook, which is not
always the same Python as a `!pip` in your shell. That mismatch is the single most common cause of
"I installed it and it still says ModuleNotFoundError".

In [ ]:
%pip install -q mermaid-py

In [ ]:
from mermaid import Mermaid


def mermaid(diagram: str) -> Mermaid:
    """Render a mermaid diagram in the notebook.

    Call it with a diagram string and make it the last line of a cell:

        mermaid("graph TD\n  A --> B")

    Jupyter displays whatever the last line of a cell evaluates to, and a
    Mermaid object knows how to draw itself as an SVG. The SVG is saved into
    the notebook's output, so once this file is committed the diagrams are
    visible to anyone reading it — no network needed.
    """
    return Mermaid(diagram)


print("mermaid() ready")

## 2. What we mean by "AI" — and why this course is about LLMs

"AI" is used to mean everything from a spam filter to a robot. Before we build anything, it is worth
being precise about which part of it we are standing in, because the choice explains the whole
syllabus.

The terms nest. Each ring below is a subset of the one outside it:

In [ ]:
mermaid("""
graph TD
    AI["<b>Artificial Intelligence</b><br/>any machine doing something<br/>we'd call intelligent"]
    ML["<b>Machine Learning</b><br/>learns patterns from data<br/>instead of hand-written rules"]
    DL["<b>Deep Learning</b><br/>neural networks with many layers"]
    GEN["<b>Generative AI</b><br/>produces new content:<br/>text, images, audio, video, code"]
    LLM["<b>Large Language Models</b><br/>generative models trained on text<br/><i>— this course lives here —</i>"]

    AI --> ML --> DL --> GEN --> LLM

    style AI fill:#f2faf6,stroke:#5b6b77
    style ML fill:#eaf7fc,stroke:#59c1e8
    style DL fill:#e9f6f4,stroke:#2f9e8f
    style GEN fill:#e7f6ee,stroke:#2e9e5b
    style LLM fill:#2e9e5b,stroke:#14212e,color:#ffffff
""")

Read it outside-in:

- **Artificial Intelligence** is the umbrella — a chess engine from 1997 counts, and it contains no
  learning at all.
- **Machine Learning** narrows it to systems that *learn* from data. Your bank's fraud detector is
  ML. So is the model that predicts house prices from a spreadsheet.
- **Deep Learning** narrows it again to neural networks deep enough to learn their own features. This
  is what made image recognition work around 2012.
- **Generative AI** is deep learning aimed at *producing* things rather than labelling them. A
  classifier answers "is this spam?"; a generative model writes the email.
- **Large Language Models** are generative models trained on enormous amounts of text.

### Why the LLM branch, specifically

Because of a happy accident: it turns out that a model good enough at predicting text becomes a
general-purpose interface to *reasoning about* text. And an astonishing share of business work is
text — tickets, policies, invoices, emails, chat logs, contracts, code, product descriptions,
reviews, specifications.

That is the pragmatic case for this course. With one skill — driving an LLM well — you can build:

| Application | What it really is underneath |
|---|---|
| Customer-support assistant (**our project**) | classify → look up → draft → escalate |
| Document Q&A over a company handbook | retrieve the right paragraph → answer with a citation |
| Extracting structured data from invoices or CVs | messy input → strict JSON |
| Code assistant, code review, migration tooling | text in, text out, where the text is code |
| Meeting summariser, report generator | long input → short structured output |
| Search that understands the question | embed the question, find near neighbours |
| A workflow of specialists that hand work to each other | several of the above, coordinated |

Every one of those is built from the same handful of moves. You will have written all of them by
session 20.

### "Language" no longer means only text

The name is now half wrong. The current generation of these models is **multimodal**: the same model
accepts images, audio, video and PDFs alongside text, and can produce images and audio as well as
words. That matters for ShopWise almost immediately — customers attach photos of broken products,
and you will hand one to the model in §7.3 tonight.

So: the branch is LLMs, the models are multimodal, and the skill is knowing how to wire them into
software that behaves reliably. That last part is what the other fifteen sessions are for.

## 3. Where we're going

One project, twenty sessions. Each session adds one capability and ends by naming the limitation the
*next* session fixes. This is the map — you will see it again in session 10 and on demo day.

In [ ]:
mermaid("""
graph LR
    A["Raw SDK calls<br/>S2–S3 · v0.1–v0.2"] --> B["LangChain<br/>one interface<br/>S4 · v1"]
    B --> C["Agent = model + tools<br/>+ memory + rails<br/>S5–S7 · v2–v4"]
    C --> D["+ Knowledge (RAG)<br/>S8–S9 · v5–v6"]
    D --> E["Explicit graphs<br/>StateGraph + interrupts<br/>S10–S11 · v7–v8"]
    E --> F["Multi-agent<br/>supervisor + specialists<br/>S12 · v9"]
    F --> G["Product<br/>PRD → API + React UI<br/>S13–S15 · v10–v12"]
    G --> H["Deployed<br/>S16 · v13"]
    H --> I["Your own model<br/>LoRA/QLoRA · eval · served<br/>S17–S18 · v14–v15"]
    I --> J["AI-engineered repo + capstone<br/>skills · hooks · MCP<br/>S19–S20 · v16+"]
""")

Every rung is a folder you can run. `labs/session-06/project/` is the assistant exactly as it stood
at the end of session 6, so you can always `diff -r session-05/project session-06/project` and see
precisely what a session added. Nothing is hidden in a git branch.

**We always build by hand before we use the shortcut.** You write an agent loop yourself in session 5
before meeting the one-line version, and a graph by hand in session 10. A shortcut you understand is
a tool; a shortcut you don't is a trap.

Now the foundation: Python. Then we start calling models.

## 4. Python for AI — the complete reference

This section assumes you know almost no Python. It is deliberately longer than any class slot,
because it has exactly one job: **for the remaining fifteen sessions, nothing here should ever be the
thing that stops you.** Every subsection ends by naming the session that needs it, so you can see
why you are learning it rather than taking it on faith.

We work through it together in class, unhurried — this is the foundation everything else stands on,
and there is no prize for finishing early. What we don't reach, you read this weekend; it is written
to be readable alone. Then come back to any part of it the moment a later session uses something
that feels unfamiliar. That is what a reference is for.

| | Topic | First needed |
|---|---|---|
| 4.1 | Reading an error message | today |
| 4.2 | Values and types | S02 |
| 4.3 | f-strings | S02 prompts |
| 4.4 | Lists | S02 |
| 4.5 | Dicts | S02 |
| 4.6 | The counting pattern | S10 graph state |
| 4.7 | Tuples and unpacking | S05 |
| 4.8 | Conditionals and truthiness | S03 |
| 4.9 | Loops and comprehensions | S03 |
| 4.10 | Functions, and why docstrings matter | S05 tools |
| 4.11 | Type hints, properly | S03 structured output |
| 4.12 | Exceptions | S03 retries |
| 4.13 | Classes | S07 middleware |
| 4.14 | Decorators — what `@` does | S05 `@tool` |
| 4.15 | Modules and imports | S02 `config.py` |
| 4.16 | Files and paths | S08 loaders |
| 4.17 | JSON and YAML | S02 |
| 4.18 | Pydantic | S03 |
| 4.19 | Generators and `yield` | S06 streaming |
| 4.20 | `async` / `await` | S13 FastAPI |
| 4.21 | Environment variables | today |
| 4.22 | The sticking-points checklist | all term |

Three of these carry more weight than the rest, because tonight's later sections use them directly:
**4.5 dicts** (every ticket is one), **4.12 exceptions** (how a failing API call is handled instead
of crashing your notebook), and **4.18 Pydantic** (which becomes structured output in §7.2).

### 4.1 Reading an error message

This comes first because it is the single most valuable skill in the room. Beginners see a wall of red
and freeze. Experienced developers read three lines and know the answer.

Python errors are read **bottom to top**:

- the **last line** is what went wrong — the error type and message
- the **line above it** is the line of your code that did it
- everything higher is the chain of calls that got there, oldest first

Below we cause an error on purpose and print it, so the notebook survives.

In [ ]:
import traceback

ticket = {"id": "TCK-001", "subject": "Where is my order?"}

try:
    print(ticket["catgory"])          # deliberate typo: no such key
except Exception:
    print(traceback.format_exc())

In [ ]:
ticket = {"id": "TCK-001", "subject": "Where is my order?"}

try:
    print(ticket["catgory"])          # deliberate typo: no such key
except Exception as e:
    print(e)

Read it: `KeyError: 'catgory'`. A `KeyError` means *you asked a dict for a key it doesn't have*. The
message names the key, so the fix is visible — a typo. You do not need to understand the middle of a
traceback to fix most problems.

The five you will actually meet this term:

| Error | It means | Usual fix |
|---|---|---|
| `KeyError: 'x'` | dict has no key `x` | typo, or use `.get("x")` if it's optional |
| `AttributeError: 'NoneType' object has no attribute ...` | something returned `None` and you used it anyway | check the thing above it that returned nothing |
| `TypeError: ... takes 2 positional arguments but 3 were given` | wrong number of arguments | check the function signature |
| `ModuleNotFoundError: No module named 'x'` | package not installed, or wrong environment | `uv sync`, and check your kernel |
| `IndentationError` | spaces don't line up | never mix tabs and spaces |

**Copy the last line into a search engine.** That is not cheating, it is the job.

### 4.2 Values and types

Every value in Python has a type. You rarely declare it, but you must know which you have, because the
operations differ — `"3" + "4"` is `"34"`, while `3 + 4` is `7`.

`None` is Python's "nothing here". It is not zero and not an empty string; it's the absence of a value,
and it's what a function returns when it returns nothing. Most beginner bugs in this course will be a
`None` you didn't expect.

In [ ]:
str subject = "Where is my order?"      # str  — text
int urgency = 4                          # int  — whole number
float price = 129.00                       # float — decimal
bool needs_human = True                   # bool — True or False
assigned_to = None                   # NoneType — deliberately empty

for value in (subject, urgency, price, needs_human, assigned_to):
    print(f"{str(value):22} {type(value).__name__}")

print()
print(f'"3" + "4" = {"3" + "4"!r}   <- string concatenation')
print(f"  3  +  4  = {3 + 4!r}    <- arithmetic")
print(f"int('4') + 3 = {int('4') + 3}  <- convert first")

`{value!r}` in an f-string means "show me the *repr*" — the developer-facing form, with quotes around
strings. It's how you tell `4` from `"4"` when debugging, and it's worth using whenever output looks
suspiciously reasonable.

**Needed in S02**, where the model returns a string and you need a number out of it.

### 4.3 f-strings

An f-string is a string with an `f` in front, and `{expressions}` inside it get evaluated. This is how
every prompt in this course is assembled.

In [ ]:
session_number = 1.5345345
print(f"Welcome everyone to Session {session_number:.2f}")

In [ ]:
name = "Ayesha"
total = 1234.5
n_items = 3

print(f"Hello {name}, your order is {total} for {n_items} items.")

# Formatting inside the braces: .2f = two decimal places, , = thousands separator
print(f"Total: {total:,.2f}")
print(f"Padded: |{name:>12}| |{name:<12}| |{name:^12}|")

# Any expression works, not just names
print(f"Average per item: {total / n_items:.2f}")

# To print a literal brace, double it
print(f"JSON looks like {{\"key\": \"value\"}}")

# Multi-line prompts: triple quotes. This is the shape of every prompt from S02 on.
prompt = f'''You are a support assistant for ShopWise.

Customer: {name}
Order total: {total:,.2f}

Reply in one short paragraph.'''
print()
print(prompt)

One trap that will bite you in S02: **quotes inside quotes.** If your f-string uses `"` and you need
`"` inside a dict access, use different quote characters — `f"{t['id']}"` works, `f"{t["id"]}"` is a
syntax error on older Pythons and confusing on all of them.

**Needed in S02**, for every prompt you write.

### 4.4 Lists

An ordered, changeable sequence. Indexing starts at **0**, and negative indices count from the end.

In [ ]:
items = [1, 4, "3kg", "2.5L", 7.5, 4, 7, 2]
items[1:-2] # items[start:end] -> end index excluding

In [ ]:
items.index("2.5L")

In [ ]:
categories = ["order_status", "refund", "warranty", "billing"]

print(f"first:  {categories[0]}")
print(f"last:   {categories[-1]}")
print(f"first two: {categories[:2]}")     # a slice: up to but NOT including index 2
print(f"length: {len(categories)}")

categories.append("shipping")             # changes the list in place
print(f"after append: {categories}")

print(f"is 'refund' in there? {'refund' in categories}")
print(f"position of 'warranty': {categories.index('warranty')}")

# Slicing does not modify; it returns a new list
middle = categories[1:3]
print(f"middle = {middle}, original still {len(categories)} long")

The off-by-one that catches everyone: `categories[:2]` gives you items 0 and 1, **not** 0, 1 and 2.
Read a slice as "from, up to".

**Needed everywhere** — a list of tickets, a list of messages, a list of retrieved documents.

### 4.5 Dicts

A dict maps **keys** to **values**. It is the single most important structure in this course: a ticket
is a dict, a message is a dict, an API response is a dict, and a graph's state in S10 is a dict.

In [ ]:
shopping_list = {"rice": 5, "beaf": 2, "oil": {"soyabean":2, "mustard": 1}}
shopping_list

In [ ]:
shopping_list.get("potato", "Potato isn't avaialble")
# shopping_list["potato"]
for item, quantity in shopping_list.items():
    print(f"Item: {item}, Quantity: {quantity}")

In [ ]:
ticket = {
    "id": "TCK-001",
    "subject": "Where is my order?",
    "category": "order_status",
    "urgency": 3,
}

print(ticket["id"])                       # by key — KeyError if absent
print(ticket.get("assignee"))             # .get — None if absent, no error
print(ticket.get("assignee", "unassigned"))   # .get with a fallback

ticket["urgency"] = 5                     # update
ticket["notes"] = "customer called twice"  # add
del ticket["notes"]                       # remove

print()
print(f"keys:   {list(ticket.keys())}")
print(f"values: {list(ticket.values())}")

print()
for key, value in ticket.items():         # the standard way to walk a dict
    print(f"  {key:10} = {value}")

**`[]` versus `.get()` is a real decision, not a style choice.** Use `[]` when a missing key means your
program is broken and should stop loudly. Use `.get()` when absence is legitimate. In S05 you will
write tools that return `{"error": "not found"}`, and reading those with `[]` will crash your agent —
that's the bug this paragraph is trying to prevent.

Dicts nest, and nesting is where beginners get lost:

In [ ]:
response = {
    "candidates": [
        {"content": {"parts": [{"text": "Your order is on the way."}]}}
    ],
    "usage": {"input": 30, "output": 27},
}

# Read it left to right: candidates -> first item -> content -> parts -> first -> text
text = response["candidates"][0]["content"]["parts"][0]["text"]
print(text)
print(f"input tokens: {response['usage']['input']}")

That chain is exactly the shape of a raw Gemini response. The SDK hands you `response.text` so you
don't have to write it — but when something is `None` and you need to know why, you will come back and
walk the chain by hand.

**Needed in S02 onward, constantly.**

### 4.6 Counting into a dict — the pattern that returns in S10

`counts.get(key, 0) + 1` reads as "whatever is there, or zero if nothing, plus one". Memorise this
shape. In session 10 it is literally how a graph node updates state.

In [ ]:
from pathlib import Path
Path.cwd().parent / "data" / "tickets.yml"

In [ ]:
import yaml
from pathlib import Path

TICKETS_PATH = Path.cwd().parent / "data" / "tickets.yml"
tickets = yaml.safe_load(TICKETS_PATH.read_text(encoding="utf-8"))
counts: dict[str, int] = {}
for t in tickets:
    counts[t["category"]] = counts.get(t["category"], 0) + 1

for category, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"{category:18} {'█' * n} {n}")

`sorted(..., key=lambda kv: -kv[1])` sorts by the second element of each pair, negated, i.e. biggest
first. A `lambda` is a one-line function with no name; you will see them used exactly like this and
almost never anywhere else in this course.

### 4.7 Tuples and unpacking

A tuple is a fixed sequence in round brackets. Its real use is returning more than one thing from a
function, and taking them apart in one line.

In [ ]:
def split_reference(ref: str) -> tuple[str, str]:
    '''Split "ORD-5001" into its prefix and number.'''
    prefix, number = ref.split("-")
    return prefix, number


kind, num = split_reference("ORD-5001")     # unpacking
print(f"kind={kind} num={num}")

# Unpacking also works in a for loop over pairs
pairs = [("order_status", 3), ("refund", 3), ("warranty", 2)]
for name, n in pairs:
    print(f"  {name}: {n}")

# enumerate gives you (index, item) pairs — the right way to number a loop
for i, t in enumerate(tickets[:3], start=1):
    print(f"{i}. {t['id']}")

Never write `for i in range(len(tickets))` and then `tickets[i]`. Use `enumerate`. It is shorter and it
cannot go out of bounds.

**Needed in S05**, where tools return pairs, and in S09 where you number citations.

### 4.8 Conditionals and truthiness

`if` / `elif` / `else`, and Python's notion of "empty means false".

In [ ]:
urgency = 4

if urgency >= 5:
    label = "critical"
elif urgency >= 3:
    label = "normal"
else:
    label = "low"
print(f"urgency {urgency} -> {label}")

# The one-line form (a "ternary") — very common in this course
label2 = "critical" if urgency >= 5 else "normal"
print(label2)

# Truthiness: these are ALL false
for value in [None, False, 0, "", [], {}]:
    print(f"  bool({value!r}) = {bool(value)}")

This is why `if not response.text:` is the idiomatic way to check for an empty answer — it catches both
`None` and `""`. But beware the flip side: `if urgency:` is `False` when urgency is `0`, which is a
real value. When zero is meaningful, test `if urgency is not None:` instead.

**Needed in S03**, where you branch on whether the model filled a field.

### 4.9 Loops and comprehensions

A comprehension builds a list in one line. Read it middle-first: *for each t in tickets, keep the ones
where …, and give me this*.

In [ ]:
# The long form
urgent = []
for t in tickets:
    if t["category"] == "complaint":
        urgent.append(t["id"])

# The same thing as a comprehension
urgent2 = [t["id"] for t in tickets if t["category"] == "complaint"]

print(urgent)
print(urgent2)
print(f"same? {urgent == urgent2}")

# Dict comprehension: {key: value for ...}
subjects = {t["id"]: t["subject"] for t in tickets[:3]}
print()
for k, v in subjects.items():
    print(f"  {k}: {v}")

# any() and all() answer yes/no questions about a whole list
print()
print(f"any shouting? {any(t['body'].isupper() for t in tickets)}")
print(f"all have ids?  {all('id' in t for t in tickets)}")

# while + break, for when you don't know how many times (S03 retries)
attempts = 0
while True:
    attempts += 1
    if attempts >= 3:
        print(f"gave up after {attempts} attempts")
        break

Don't nest comprehensions more than one level. If it needs two, write the loop — you will read this
code again in week 7.

**Needed in S03** (retry loops) **and S08** (splitting documents).

### 4.10 Functions, and why docstrings matter more here than anywhere else

A function takes inputs, does work, returns a result. Parameters can have defaults; arguments can be
passed by position or by name.

Then the part that is unusual about this course: **in session 5 your docstrings get read by the
model.** When you write a tool, the model decides whether to call it based on the docstring you wrote.
A vague docstring is a bug that manifests as the AI doing the wrong thing. This is the only field in
software where prose is executable, so start writing them properly now.

In [ ]:
def triage(text: str, default_urgency: int = 3, *, notify: bool = False) -> dict:
    '''Classify a support ticket.

    Args:
        text: the customer's message, as they wrote it.
        default_urgency: used when the message gives no signal, 1-5.
        notify: whether a human should be paged immediately.

    Returns:
        A dict with keys "category", "urgency" and "needs_human".
    '''
    shouting = text.isupper()
    return {
        "category": "complaint" if shouting else "general",
        "urgency": 5 if shouting else default_urgency,
        "needs_human": shouting or notify,
    }


print(triage("MY ORDER NEVER ARRIVED"))
print(triage("Hello, quick question about warranty"))
print(triage("Hello", default_urgency=1))       # by name — clearer at the call site
print(triage("Hello", notify=True))             # after *, name is REQUIRED

The `*` in the signature means everything after it must be passed by name. It exists so that
`triage(text, True)` — which reads like nothing — becomes impossible. You will see this in library
signatures all term, including `client.files.upload(file=...)`.

**Needed in S05.** Read that docstring above again and imagine an AI deciding from it alone.

### 4.11 Type hints, properly

Hints are promises to readers and tools; Python does not enforce them at runtime. They matter here
because from S03 onward **the same hints tell the model what shape to reply in** — a Pydantic model's
annotations become the JSON schema sent to Gemini.

In [ ]:
from typing import Literal, Optional

# Built-in generics — the modern syntax, no imports needed
def ids_of(items: list[dict]) -> list[str]:
    return [i["id"] for i in items]

# "str or nothing" — two equivalent spellings, prefer the first
def find_subject(ticket_id: str) -> str | None:
    for t in tickets:
        if t["id"] == ticket_id:
            return t["subject"]
    return None                       # explicit: not found

def find_subject_old(ticket_id: str) -> Optional[str]:   # same thing, older style
    return find_subject(ticket_id)

# Literal pins the allowed values — this is what stops the model inventing
# "billing issue", "Billing" and "BILLING" for the same category in S03.
Category = Literal["order_status", "refund", "warranty", "billing", "other"]

def label(category: Category) -> str:
    return category.replace("_", " ").title()

print(ids_of(tickets[:3]))
print(find_subject("TCK-002"))
print(find_subject("TCK-999"))         # None, not a crash
print(label("order_status"))

`Literal` is the quiet hero of session 3. Free-text categories drift — run the same ticket three times
and you get three spellings. `Literal` turns the field into a closed set, and the model complies.

**Needed in S03**, and every session after it.

### 4.12 Exceptions

Things fail: the network, the API, a rate limit, a malformed reply. `try` / `except` lets you decide
what happens instead of the program stopping.

In [ ]:
def parse_urgency(raw: str) -> int:
    '''Turn the model's answer into an int, falling back to 3.'''
    try:
        value = int(raw)
    except ValueError:                       # catch the SPECIFIC error
        print(f"  could not parse {raw!r}, using default")
        return 3
    else:                                    # runs only if no exception
        return max(1, min(5, value))         # clamp into 1..5
    finally:                                 # runs either way
        pass


print(parse_urgency("4"))
print(parse_urgency("very urgent"))

# Raising your own, when a caller has done something impossible
def get_ticket(ticket_id: str) -> dict:
    for t in tickets:
        if t["id"] == ticket_id:
            return t
    raise KeyError(f"no ticket {ticket_id}")


try:
    get_ticket("TCK-999")
except KeyError as exc:
    print(f"handled: {exc}")

**Never write a bare `except:` or `except Exception: pass`.** It hides the bug you most need to see —
including your own typos — and turns a five-minute fix into an afternoon. Catch the specific error you
expect, and let anything else crash loudly.

**Needed in S03** for retries, and in S02 when your first API call fails.

### 4.13 Classes

A class bundles data with the operations on that data. `__init__` runs when you create one, and `self`
is the particular instance being worked on.

In [ ]:
class Ticket:
    def __init__(self, id: str, subject: str, body: str, category: str) -> None:
        self.id = id                 # these are "attributes" on the instance
        self.subject = subject
        self.body = body
        self.category = category

    def is_shouting(self) -> bool:
        letters = [ch for ch in self.body if ch.isalpha()]
        if not letters:
            return False
        return sum(ch.isupper() for ch in letters) / len(letters) > 0.7

    def __repr__(self) -> str:
        '''What you see when you print the object. Without it: <Ticket at 0x10f...>.'''
        return f"Ticket({self.id}, {self.category})"


# **t unpacks a dict into keyword arguments. It works ONLY because the dict's
# keys match the parameter names, which is why 'from' and 'received' are dropped.
objects = [
    Ticket(id=t["id"], subject=t["subject"], body=t["body"], category=t["category"])
    for t in tickets
]

print(objects[:3])
print(f"shouting: {[t for t in objects if t.is_shouting()]}")

In [ ]:
# Inheritance: a subclass reuses and extends a parent. This is the shape of
# session 7's middleware, where you subclass a base class the framework provides.
class UrgentTicket(Ticket):
    def __init__(self, id: str, subject: str, body: str, category: str, sla_hours: int = 4):
        super().__init__(id, subject, body, category)    # run the parent's __init__
        self.sla_hours = sla_hours

    def __repr__(self) -> str:
        return f"UrgentTicket({self.id}, SLA {self.sla_hours}h)"


u = UrgentTicket("TCK-003", "shouting", "HELP ME NOW", "complaint")
print(u)
print(f"still a Ticket? {isinstance(u, Ticket)}")
print(f"inherited method works: {u.is_shouting()}")

**Needed in S07**, where guardrails are classes you subclass, and in S03 where Pydantic models are
classes.

### 4.14 Decorators — what `@` actually does

You will type `@tool` in session 5 and `@app.post` in session 13. Most people use decorators for years
without knowing what they are, and then get stuck the first time one behaves oddly. It is simpler than
it looks.

A decorator is **a function that takes your function and gives back a replacement.** That's all.
`@shout` above `def greet` means exactly `greet = shout(greet)`.

In [ ]:
import functools


def announce(func):
    '''A decorator: wraps func so every call prints what it is doing.'''

    @functools.wraps(func)                 # keeps func's name and docstring intact
    def wrapper(*args, **kwargs):          # accepts ANY arguments and passes them on
        print(f"  -> calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"  <- {func.__name__} returned {result!r}")
        return result

    return wrapper


@announce
def add_urgency(a: int, b: int) -> int:
    '''Add two urgency scores.'''
    return a + b


print("with the decorator:")
add_urgency(2, 3)

# Proof that @ is just assignment:
def plain(x): return x * 2
plain = announce(plain)
print("\nmanually decorated:")
plain(21)

print(f"\nname survived @functools.wraps: {add_urgency.__name__}")
print(f"docstring survived: {add_urgency.__doc__}")

`*args` collects positional arguments into a tuple; `**kwargs` collects named ones into a dict. Together
they mean "whatever you were given, pass it along" — which is why every decorator looks like this.

Now `@tool` in session 5 is demystified: it takes your plain Python function, reads its name, its type
hints and its docstring, and returns an object the model can be told about. You already wrote the hard
part in 4.10.

**Needed in S05, S07 and S13.**

### 4.15 Modules and imports

Every `.py` file is a module. `import x` gives you `x.thing`; `from x import thing` gives you `thing`
directly.

From session 2 there is a `config.py` beside each session's code holding the model IDs. That is the
whole reason imports matter here: **one place to change, everywhere updated.**

In [ ]:
# Three import styles, all used in this course
import json                          # json.dumps(...)
from pathlib import Path             # Path(...)
import yaml as y                     # y.safe_load(...)  — aliasing, used sparingly

print(json.dumps({"ok": True}))
print(Path.cwd().name)
print(type(y.safe_load("a: 1")).__name__)

# What a config.py looks like, and why os.getenv is in it:
import os
MODEL_DEMO = os.getenv("SHOPWISE_MODEL", "gemini-3.1-flash-lite")
print(f"\nMODEL from env, or the default: {MODEL_DEMO}")

`os.getenv("NAME", "fallback")` reads an environment variable and uses the fallback if it isn't set.
That one line is why a model retirement in 2027 is a single edit to `labs/.env` rather than fifteen
edits across fifteen frozen session folders.

The `if __name__ == "__main__":` guard at the bottom of `warmup.py` means "only run this when the file
is executed directly, not when it's imported". Without it, importing a module runs everything in it,
which is a surprising and annoying way to lose an afternoon.

**Needed from S02 onward.**

### 4.16 Files and paths

`pathlib` is the modern way. Never build a path by gluing strings with `/` — it breaks on Windows.

In [ ]:
from pathlib import Path

data_dir = Path.cwd().parent / "data"        # the / operator joins path parts
print(f"data dir: {data_dir}")
print(f"exists:   {data_dir.exists()}")
print(f"contents: {[p.name for p in data_dir.iterdir()]}")

f = data_dir / "tickets.yml"
print(f"\nname={f.name}  suffix={f.suffix}  parent={f.parent.name}")
print(f"size: {len(f.read_text(encoding='utf-8')):,} characters")

# Writing — into a scratch file we then remove
scratch = Path("_scratch_demo.txt")
scratch.write_text("hello from session 1\n", encoding="utf-8")
print(f"\nwrote and read back: {scratch.read_text(encoding='utf-8').strip()!r}")
scratch.unlink()                              # delete
print(f"cleaned up, exists now: {scratch.exists()}")

Always pass `encoding="utf-8"` explicitly. The default differs between operating systems, and TCK-004
is in Bangla — this is the line that stops it arriving as mojibake on someone's Windows laptop.

**Needed in S08**, loading the policy documents.

### 4.17 JSON and YAML

Two text formats for structured data. **JSON** is what APIs speak. **YAML** is friendlier for humans to
write, so this course uses it for fixtures and prompts.

In [ ]:
import json
import yaml

# JSON: dict -> string -> dict
as_text = json.dumps(tickets[3], ensure_ascii=False, indent=2)
print(as_text)

back = json.loads(as_text)
print(f"\nround trip intact: {back == tickets[3]}")

# YAML: the same data, in the format humans edit
print()
print(yaml.safe_dump(tickets[0], allow_unicode=True, sort_keys=False))

Three details that cause real bugs:

- **`ensure_ascii=False`** keeps Bangla readable instead of escaping it into `\uXXXX`.
- **`yaml.safe_load`, never `yaml.load`.** Plain `load` can execute arbitrary Python from the file. That
  is a genuine vulnerability, not a theoretical one.
- **YAML guesses types.** `received: 2026-08-02` unquoted becomes a `date` object, and `json.dumps`
  then refuses to serialise it. That is why every date in `tickets.yml` is quoted — we hit this exact
  error while writing this course.

**Needed in S02**, where prompts live in YAML and the API replies in JSON.

### 4.18 Pydantic — a class that checks itself

Session 3's big idea. A Pydantic model looks like a class with type hints, but it **validates** data at
runtime, and it can hand its schema to Gemini so the model replies in exactly that shape.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError


class TriageResult(BaseModel):
    '''What the assistant must return for every ticket. Born here, used until S20.'''
    category: Literal["order_status", "refund", "warranty", "billing", "other"]
    urgency: int = Field(ge=1, le=5, description="1 = trivial, 5 = drop everything")
    needs_human: bool = False


ok = TriageResult(category="refund", urgency=5, needs_human=True)
print(ok)
print(f"as a dict: {ok.model_dump()}")
print(f"field access: {ok.urgency}")

# Now break it on purpose — this is the value of Pydantic
print()
try:
    TriageResult(category="billing issue", urgency=9)
except ValidationError as exc:
    print(f"{exc.error_count()} problems found:")
    for e in exc.errors():
        print(f"  {e['loc'][0]}: {e['msg']}")

Look at what the errors caught: an invented category (`"billing issue"` is not in the `Literal`) and an
out-of-range urgency (`9` fails `le=5`). In session 3 those same rules are sent *to the model* as a JSON
schema, so it fills them correctly in the first place — and if it doesn't, you find out immediately
instead of three layers downstream.

`Field(description=...)` is not decoration: the description travels to the model as part of the schema.
Another place where your prose is executable.

**Needed in S03**, and `TriageResult` survives to session 20.

### 4.19 Generators and `yield`

A function with `yield` doesn't return once — it produces values one at a time, on demand. This is how
streaming works: session 6 prints an answer token by token instead of waiting for the whole thing.

In [ ]:
def word_stream(text: str):
    '''Yield one word at a time instead of returning them all at once.'''
    for word in text.split():
        yield word


gen = word_stream("your order is on the way")
print(f"a generator, not a list: {type(gen).__name__}")

for word in gen:
    print(word, end=" ")
print()

# It is consumed — a second loop gets nothing. A common surprise.
print(f"second pass: {list(gen)}")

# Wrap in list() if you genuinely need it all at once
print(f"fresh, as a list: {list(word_stream('one two three'))}")

`for event in stream:` in §7.6 of this very notebook is exactly this loop — the streaming call
returns a generator, and each `next()` resumes the API's reader where it left off. The reason
streaming feels fast is not that the model is faster; it's that you stopped waiting for the last
token before showing the first.

**Used tonight in §7.6**, properly in S06, and again in S13 for server-sent events.

### 4.20 `async` and `await`, the minimum

Session 13 serves the assistant over HTTP with FastAPI, and its handlers are `async`. You need to
recognise three things, not master concurrency.

`async def` defines a coroutine. Calling it does **not** run it — it returns an object you must
`await`. `await` means "pause here, let other work proceed, resume when this finishes". It only works
inside `async def`, or at the top level of a notebook, which is what the cell below relies on.

In [ ]:
import asyncio


async def fetch_order(order_id: str) -> dict:
    '''Pretend to call a slow backend.'''
    await asyncio.sleep(0.1)                 # a real call would be a network wait
    return {"order_id": order_id, "status": "shipped"}


# Calling without await gives you the coroutine, not the answer:
pending = fetch_order("ORD-5001")
print(f"without await: {type(pending).__name__}")

result = await pending                        # top-level await works in notebooks
print(f"with await:    {result}")

# The payoff: three waits at once instead of one after another
both = await asyncio.gather(
    fetch_order("ORD-5001"),
    fetch_order("ORD-5002"),
    fetch_order("ORD-5003"),
)
print(f"gathered {len(both)} results concurrently: {[r['order_id'] for r in both]}")

The error you will meet is `RuntimeWarning: coroutine ... was never awaited`, which means you forgot
`await`. In a plain `.py` script you would need `asyncio.run(main())` at the bottom, because there is no
event loop yet — in a notebook one is already running, which is why `asyncio.run` fails here but
top-level `await` works.

**Needed in S13.** Until then you can forget this section exists.

### 4.21 Environment variables

A variable that lives in the operating system rather than your code, so secrets never get committed.
`.env` + `python-dotenv` is the convention this course uses.

In [ ]:
import os

print(f"key set: {bool(os.getenv('GEMINI_API_KEY'))}")
print(f"missing var with fallback: {os.getenv('NOT_SET_ANYWHERE', 'fallback value')}")

# Never do this:
#   API_KEY = "AIzaSy..."      <- committed, leaked, revoked, and it was your fault
# Always this:
#   API_KEY = os.getenv("GEMINI_API_KEY")

key = os.getenv("GEMINI_API_KEY", "")
print(f"\nsafe to show: {key[:6]}...{key[-4:] if key else ''} ({len(key)} chars)")

That last line is the only acceptable way to print anything about a key: first few characters, last few,
length. Enough to tell two keys apart, not enough to use.

**Needed today**, and every session after.

### 4.22 The sticking-points checklist

When something breaks this term, it is almost always one of these. Come back here first.

| Symptom | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError` | wrong environment or kernel | `uv sync` in `labs/`; then pick the `.venv` kernel in Jupyter |
| `No API key was provided` | `.env` not found, or key empty | `load_dotenv` searches from the *calling file's* folder, not cwd — pass the path |
| `KeyError` | typo, or an optional field | `.get("x")` when absence is legitimate |
| `AttributeError: 'NoneType' has no attribute …` | something above returned `None` | print that thing; a `.get()` or a failed lookup is usual |
| `429 RESOURCE_EXHAUSTED` | rate limit, not your code | wait a minute; check your own limits in AI Studio |
| `TypeError: Object of type date is not JSON serializable` | YAML turned a date into an object | quote it in the YAML, or `default=str` |
| Bangla prints as `?????` or mojibake | missing encoding | `encoding="utf-8"` on every read and write |
| `coroutine was never awaited` | forgot `await` | add it; `asyncio.run` in scripts, top-level `await` in notebooks |
| `IndentationError` | tabs mixed with spaces | four spaces, never tabs |
| Notebook variable is stale | cells run out of order | Kernel → Restart and Run All before you believe anything |

That last one deserves emphasis. A notebook remembers everything you have run, in the order you ran it.
If results stop making sense, **restart the kernel and run from the top** before debugging anything else.

### The warm-up script

`warmup.py` in this folder is this same material as a runnable script rather than a notebook — worth
running once, because a `.py` file is what every session from S02 onward actually ships:

```bash
uv run python warmup.py
```

It ends with a `# Your turn` comment. **That comment is exercise ⭐⭐**, and the pattern it asks for
(§4.6's counting loop) reappears in session 10 as the way a graph updates its state.

## 5. Setup: your key, and the client

Two things happen in the cell below. `load_dotenv()` reads `labs/.env` and puts your key into the
environment, and `genai.Client()` picks it up from there. **Your key never appears in code** — that
is the entire point of a `.env` file. Code gets committed and shared; `.env` is gitignored and stays
on your machine.

The trap worth knowing now, because it will cost someone an evening otherwise: `load_dotenv()` with
no arguments does **not** search from your current working directory. It searches upward from the
directory of the file that called it. From this notebook that happens to work — `labs/session-01/` is
one level below `labs/.env`. From a script somewhere else it silently finds nothing, and you get
`No API key was provided` while staring at a `.env` file that obviously exists. So we pass the path
explicitly, and we print whether it worked.

If you don't have a key yet: `https://aistudio.google.com/apikey`, free, no card required. Paste it
into `labs/.env` **without quotes around it** — a pasted `"AIza..."` including the quote characters
is the second most common setup failure.

In [ ]:
!pip install google-genai==2.19.0

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google import genai

# labs/session-01/ -> labs/.env   (explicit beats magic)
ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(ENV_PATH)

print(f"reading  {ENV_PATH}")
print(f"key found: {bool(os.getenv('GEMINI_API_KEY'))}")

# Model IDs live in ONE place, always. From session 2 that place is config.py.
# A model ID scattered through thirty cells is a migration you do by hand at 2am
# on the day Google retires it.
MODEL = "gemini-3.1-flash-lite"          # course default: fast and cheap
MODEL_IMAGE = "gemini-3.1-flash-image"   # image generation (§7.5)

client = genai.Client()                  # reads GEMINI_API_KEY from the environment
print(f"client ready, model = {MODEL}")

### One helper, so that a failure teaches instead of frightening

Some cells below use capabilities that a billing-enabled key unlocks. On a free key they raise. An
uncaught red traceback on day one is discouraging and tells a beginner nothing, so we write a small
function that turns an exception into a sentence you can act on, and use it wherever a call can
legitimately fail.

This is also a first look at the pattern §4.12 described: catch the *specific* thing you expect, say
what happened, and carry on.

In [ ]:
def explain_error(err: Exception) -> str:
    """Turn an API exception into one actionable sentence.

    Called from the `except` block of every cell that can legitimately fail on a
    free key, so the notebook explains itself instead of showing a traceback.
    """
    text = str(err)

    if "RESOURCE_EXHAUSTED" in text or "429" in text:
        return ("Rate limit, not a bug in your code. Wait a minute and re-run. "
                "Your own limits are at https://aistudio.google.com/rate-limit")
    if "403" in text or "PERMISSION_DENIED" in text or "billed" in text.lower():
        return ("This capability needs a billing-enabled key. The committed output "
                "above shows what it returns — nothing in this course requires you "
                "to enable billing.")
    if "404" in text or "NOT_FOUND" in text:
        return ("That model ID no longer exists. Models get retired; run the "
                "listing cell in §9.6 to see what is available today.")
    if "API key" in text:
        return "No key was found. Check labs/.env and re-run the setup cell above."
    return f"Unexpected — read it carefully, it usually says what is wrong:\n{text[:400]}"


print("explain_error() ready")

## 6. Your first call to a modern model

Three lines. Read them, then run them.

```python
interaction = client.interactions.create(model=MODEL, input="…")
print(interaction.output_text)
```

**A note on which API this is, because you will see two in the wild.** Google currently offers two
ways to call these models:

| | `client.models.generate_content(...)` | `client.interactions.create(...)` |
|---|---|---|
| Status | the long-standing, stable surface | the newer **Interactions API**, currently in Beta |
| Shape | you resend the whole conversation every time | the API can hold the conversation for you |
| Returns | text and parts | a *timeline of steps* — thoughts, searches, tool calls, output |
| Used by | LangChain, which we adopt in session 4 | this notebook |

We use the Interactions API tonight because it shows the model's capabilities with the least
ceremony, and because its step timeline makes visible something we will spend sessions 5 and 10
building by hand — the fact that a model *does things in a sequence*, not in one shot. Google's own
recommendation for production today is still `generate_content`, and that is what LangChain wraps, so
you will meet both. Neither is wasted knowledge.

In [ ]:
interaction = client.interactions.create(
    model="gemini-3.1-flash-lite",
    input="Think very deeply and concisely give us 3 benifits to adopt AI to any e-commerce business for theri customer support",
)

print(interaction.output_text)

### What actually came back

`output_text` is a convenience. The object underneath has more in it, and looking at it once now
means the vocabulary of the rest of the course is already familiar.

In [ ]:
interaction.usage

In [ ]:
print(f"id      : {interaction.id}")
print(f"model   : {interaction.model}")
print(f"status  : {interaction.status}")
print(f"steps   : {[s.type for s in interaction.steps]}")
print()
print(f"tokens in  : {interaction.usage.total_input_tokens}")
print(f"tokens out : {interaction.usage.total_output_tokens}")
print(f"'thinking' : {interaction.usage.total_thought_tokens}")
print(f"total      : {interaction.usage.total_tokens}")

Three things to notice, all of which return later:

1. **`steps` is a list.** A simple question produces something like
   `['thought', 'model_output']` — the model thought, then answered. When we give it a search tool in
   §7.4 you will see a search step appear in that same list. This is the raw material of agents.
2. **`id`.** The API stored this exchange. §7.7 uses that id to continue the conversation without
   resending anything.
3. **`usage`.** Every call reports what it consumed, in both directions. §8 is about that number, and
   from session 4 onward LangSmith records it for you automatically on every call you make all term.

## 7. The capability tour

Seven calls, all pointed at ShopWise's actual problems. The goal is not to teach each API in depth —
each one gets its own session later — but to establish what is *possible*, so that when we spend
session 8 on retrieval you already know why.

Every example uses the same client and the same three-line shape. What changes is what we put in, and
what we ask for back.

### 7.1 Text in, text out — triaging a ticket

The job a human does now: read the ticket, decide what it is about, decide how urgent it is, suggest
a first move. Let's take the worst one in the inbox — TCK-003, the customer writing in capitals for
the third time.

`tickets` was loaded back in §4.6, so it is already in memory.

In [ ]:
ticket = tickets[2]   # TCK-003 — the one in capitals

interaction = client.interactions.create(
    model=MODEL,
    input=(
        "You are a support triage assistant for an online store.\n"
        "Reply with the category, an urgency from 1 to 5, and write one line response.\n\n"
        f"Subject: {ticket['subject']}\n{ticket['body']}"
    ),
)

print(f"TICKET  {ticket['id']}: {ticket['subject']}\n")
print(interaction.output_text.strip())

That took about a second. A human doing it carefully takes two to four minutes, and ShopWise gets
around two hundred tickets a day.

But look closely at what came back and you will find the problem that defines session 3: **it is
prose.** You cannot store prose in a database, sort by it, route on it, or count it. If the model
writes "Urgency: 5" today and "urgency is high" tomorrow, the code downstream breaks. Which is
exactly what the next cell fixes.

### 7.2 Structured output — JSON your code can actually use

We describe the shape we want with a Pydantic model (§4.18), hand its JSON schema to the API, and get
back something guaranteed to parse. `Literal` is doing real work here: it does not merely *suggest*
the categories, it constrains the output to exactly those strings, so a free-text category can never
drift into your database.

This is the single most useful trick in applied LLM work, and it is why session 3 exists.

In [ ]:
from pydantic import BaseModel
class SimpleCalc(BaseModel):
    sum: float
    multiplication: float

In [ ]:
result = SimpleCalc(sum=35, multiplication=4.6)

In [ ]:
result.sum

In [ ]:
def test(a: int, b: float) -> SimpleCalc:
    sum = a + b
    multiplication = a * b

    return SimpleCalc(sum=sum, multiplication=multiplication)

In [ ]:
result = test(3, 2.5)
result

In [ ]:
result.multiplication

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class Triage(BaseModel):
    """The shape ShopWise wants every ticket reduced to."""
    # These are exactly the categories used in data/tickets.yml, so the model's
    # answer can be compared against the hand-labelled ground truth. From S03
    # that comparison becomes a real evaluation.
    category: Literal[
        "billing", "complaint", "order_status", "other", "product_question",
        "refund", "return", "shipping", "warranty",
    ]
    urgency: int = Field(ge=1, le=5, description="1 = can wait a week, 5 = answer now")
    mentions_order_id: str | None = Field(description="e.g. ORD-5002, or null if none")
    summary: str = Field(description="One line, neutral tone, no exclamation marks")


interaction = client.interactions.create(
    model=MODEL,
    input=f"Triage this support ticket.\n\nSubject: {ticket['subject']}\n{ticket['body']}",
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": Triage.model_json_schema(),
    },
)

# .output_text is now a JSON string. Pydantic parses AND validates it: if the
# model returned an urgency of 9, this line raises instead of quietly poisoning
# your data.
result = Triage.model_validate_json(interaction.output_text)

print(result)
print()
print(f"route to      : {result.category} queue")
print(f"page someone? : {result.urgency >= 4}")
print(f"order to look up: {result.mentions_order_id}")

Read those last three lines again. `result.urgency >= 4` is **ordinary Python making a decision**.
The model's answer stopped being a paragraph a human reads and became a value a program can branch
on. That is the whole transition from "a chatbot" to "software with a model inside it", and it
happened in one extra argument.

Two honest caveats, so you don't over-trust it:

- The schema constrains the *shape*, not the *truth*. The model can still return
  `category="refund"` when the ticket was about billing. Correct shape, wrong answer.
- Descriptions in `Field(...)` are sent to the model and genuinely steer it. Writing
  `"e.g. ORD-5002, or null if none"` is prompt engineering wearing a type hint's clothes.

**One forward-warning, so it doesn't ambush you in session 3.** Structured output exists on both
Google APIs, but the argument is named differently on each: it is `response_format` here on the
Interactions API, and `response_mime_type` + `response_schema` on `generate_content`, where the
parsed object comes back as `response.parsed`. Same idea, two spellings. Session 3 uses the second
one, for the reason given in §6 — LangChain is built on it, and S04's whole argument depends on
having hand-written that shape first.


### 7.3 The customer attached a photo — image understanding

Real support inboxes are full of photographs. Someone's headphones arrived damaged and they took a
picture of it rather than describing it, because that is what people do.

`data/images/ticket-photo-headphones.jpg` is a photo attached to a warranty claim. We pass it to the
model alongside a question. Note the shape of `input`: instead of a single string it is now a **list
of content blocks**, one text and one image. That list is how every multimodal request is built —
audio, video and PDF blocks work the same way.

Images are sent base64-encoded, which is just "bytes rewritten using only characters that survive
being put in JSON".

In [ ]:
import base64

from IPython.display import Image, display

PHOTO = Path.cwd().parent / "data" / "images" / "ticket-photo-headphones.jpg"

display(Image(filename=str(PHOTO), width=440))
print(f"{PHOTO.name} — {PHOTO.stat().st_size / 1024:.0f} KB")

In [ ]:
photo_b64 = base64.b64encode(PHOTO.read_bytes()).decode("utf-8")

interaction = client.interactions.create(
    model=MODEL,
    input=[
        {"type": "text", "text": (
            "A customer attached this photo to a warranty claim for ShopWise.\n"
            "1. What product is it?\n"
            "2. What specifically is damaged?\n"
            "3. Does this look like manufacturing failure or normal wear?\n"
            "Answer in three short lines. Say plainly if you cannot tell."
        )},
        {"type": "image", "data": photo_b64, "mime_type": "image/jpeg"},
    ],
)

print(interaction.output_text.strip())

Sit with that for a second: nobody wrote image-processing code. The same three-line call that triaged
text read a photograph and reasoned about wear versus manufacturing failure.

**And now the professional caution**, because this capability is the easiest one to over-trust. The
model is describing what the picture *looks like*. It cannot know how old the product is, whether the
customer sat on it, or what ShopWise's warranty actually covers. Question 3 is genuinely beyond a
photograph — watch whether it hedged appropriately or answered confidently anyway. Either way, the
architecture lesson is the same one that runs through this whole course: **a model's opinion is an
input to a decision, never the decision itself.** Session 11 is where a human signs off before money
moves.

### 7.4 Answering from the live web — search grounding with citations

💳 **Instructor demo — needs a billing-enabled key.**

A model's knowledge is frozen at whenever its training data stopped. Ask it about this morning's
news, or a competitor's current price, and it has nothing — but it will often answer anyway, because
producing a plausible continuation is what it does.

Giving it a **search tool** changes that. Watch two things in the output: the answer is current, and
every claim carries a URL you can click and check. That second part is what makes this usable in a
business setting, and it previews session 9, where we make ShopWise cite its own policy documents or
refuse to answer.

This is also the foundation of the course's [ADV] "Market Analyst" track from session 12.

In [ ]:
try:
    interaction = client.interactions.create(
        model=MODEL,
        input=(
            "ShopWise sells consumer electronics online. What return window do the major "
            "online electronics retailers currently advertise? Answer in three bullets, "
            "and name the retailer for each."
        ),
        tools=[{"type": "google_search"}],
    )

    print(interaction.output_text.strip())

    # The step timeline now contains search steps — the model decided to use the
    # tool, read the results, and only then answered.
    print(f"\nsteps taken: {[s.type for s in interaction.steps]}")

    # Citations arrive as annotations attached to the text blocks. The same source
    # is often cited for several sentences, so we de-duplicate before printing.
    sources = {}
    for step in interaction.steps:
        if step.type != "model_output":
            continue
        for block in step.content:
            for annotation in getattr(block, "annotations", None) or []:
                if annotation.type == "url_citation":
                    sources.setdefault(annotation.title, annotation.url)

    print(f"\nSources ({len(sources)}):")
    for title, url in sources.items():
        print(f"  · {title:22} {url[:58]}…")

except Exception as err:
    print("Search grounding did not run.")
    print(explain_error(err))

Notice what appeared in `steps`. On our very first call the timeline was
`['thought', 'model_output']`. Give the model a tool and the timeline grows — it now searches, reads
the results, and *then* answers. Nobody wrote that loop. We handed over one tool and the model
sequenced the work itself.

That is an **agent**, in one argument, and it is what session 5 takes apart and rebuilds by hand so
that it becomes a tool you understand rather than a trap you rely on.

Two practical notes about the citations, because both surprise people:

- **The URLs are Google redirect links**, not the publisher's address. They resolve to the real page
  when opened; Google routes them so it can attribute the traffic. If you are building a UI, show
  the `title` and let the redirect do its job.
- **The same source is cited repeatedly** — once per sentence it supported — which is why the cell
  de-duplicates before printing. A citation list that repeats "bestbuy.com" six times looks broken
  to a customer even though it is technically correct.

### 7.5 Making an image — the replacement product listing

💳 **Instructor demo — needs a billing-enabled key.**

Generation is not only text. ShopWise needs listing photography, seasonal banners and social posts,
and a shop owner in Mirpur does not have a photo studio. A different model — the same client, the
same `create` call — returns an image instead of words.

The only real difference is that the result arrives as base64 bytes on `output_image` rather than
text on `output_text`.

In [ ]:
from io import BytesIO

from PIL import Image as PILImage

try:
    interaction = client.interactions.create(
        model=MODEL_IMAGE,
        input=(
            "A clean e-commerce product photo for an online electronics store: "
            "matte black over-ear headphones on a plain light background, "
            "soft studio lighting, slight angle, no text, no logos, no branding."
        ),
    )

    generated = PILImage.open(BytesIO(base64.b64decode(interaction.output_image.data)))
    print(f"{interaction.output_image.mime_type} — {generated.size[0]}x{generated.size[1]}")
    display(generated.resize((generated.width // 2, generated.height // 2)))

except Exception as err:
    print("Image generation did not run.")
    print(explain_error(err))

Two things worth saying out loud about generated images, and neither is technical:

- **Disclose it.** An invented product photo used as if it were the real item is a
  misrepresentation, whatever the tooling. Generated *concept* imagery, banners and mockups are
  ordinary commercial work; a generated photo of the thing in the box is not.
- **This is where costs stop being trivial.** A text triage is a fraction of a paisa. Images are
  priced per image and are orders of magnitude more. §8 is about learning to look before you scale
  anything up.

### 7.6 Streaming — why every chat interface feels fast

When you use ChatGPT or Gemini, words appear as they are produced rather than after the whole answer
is finished. That is not a UI animation; it is the API handing back pieces as the model generates
them.

The total time is identical. The *perceived* time is transformed, because the reader starts reading
immediately. Session 6 wires this into ShopWise, and session 15 puts it on screen in React.

`stream=True` changes the return value from an object into something you loop over.

In [ ]:
stream = client.interactions.create(
    model=MODEL,
    input=(
        "Write a warm three-sentence apology to a ShopWise customer whose order is "
        "five days late. Do not promise a specific new delivery date."
    ),
    stream=True,
)

for event in stream:
    delta = getattr(event, "delta", None)
    # The stream carries several kinds of event — the model's internal thinking,
    # step boundaries, status updates. We want only the text pieces.
    if delta is not None and getattr(delta, "type", None) == "text":
        print(delta.text, end="", flush=True)

print()

The `getattr` dance in that loop is not ceremony. A stream carries several kinds of event —
`step.start`, the model's internal thinking, status updates, `step.stop` — and only some of them are
text you want to show a customer. Filtering to the pieces you actually want is the entire job of a
streaming handler, and it is why session 6 spends real time here.

Also note what streaming does *not* do: it is not faster, and you cannot validate a JSON schema
against half an answer. Streaming and structured output pull in opposite directions — a genuine
design trade-off you will make in session 13.

### 7.7 Remembering the conversation

A model has no memory between calls. Ask it a follow-up question with a fresh request and it has no
idea what you are talking about — which is why "the AI forgot what I said" is the most common
complaint about home-made chatbots.

There are exactly two ways to fix that, and you should know both:

1. **Resend everything.** Keep the transcript yourself and send the whole history with every request.
   Total control, and you pay for the whole conversation on every single turn.
2. **Let the API keep it.** Pass the previous interaction's `id`, and the service links them.

The Interactions API supports both. Below is the second, because it is one argument. Session 6 builds
the first by hand, and session 11 makes it survive a server restart.

In [ ]:
first = client.interactions.create(
    model=MODEL,
    input="I'm chasing order ORD-5002 — two kettles, ordered on the 1st, still not shipped.",
)
print("turn 1:", first.output_text.strip()[:220], "…\n")

second = client.interactions.create(
    model=MODEL,
    input="Which order number was that again, and what did I buy?",
    previous_interaction_id=first.id,     # <- the entire memory mechanism
)
print("turn 2:", second.output_text.strip())

It remembered. Now look at the cost of remembering:

In [ ]:
print(f"turn 1 input tokens: {first.usage.total_input_tokens}")
print(f"turn 2 input tokens: {second.usage.total_input_tokens}")
print()
print("Turn 2's question is six words. The input token count is much larger than six words,")
print("because the earlier turn was sent to the model again underneath.")

**This is the single most expensive misunderstanding in LLM applications.** A conversation is not
stored in the model. Every turn, the whole history goes back through it. A chat that runs for forty
turns is paying for turn 1 forty times over.

Nothing about `previous_interaction_id` avoids that — it saves you the bookkeeping, not the tokens.
Session 6 is where we start managing this deliberately: summarising old turns, dropping what no
longer matters, and deciding what is worth carrying forward.

## 8. What did that cost? Tokens, the unit of everything

You have now made a dozen calls. Time to understand the meter.

A model does not read characters or words. It reads **tokens** — chunks of text produced by a
tokenizer, roughly a common word or a fragment of a longer one. Everything is measured in them:

- **Cost** is per token, in both directions. You pay for what you send *and* what comes back.
- **The context window** — the maximum a model can consider at once — is measured in tokens.
- **Speed** is roughly proportional to tokens generated.

So "how many tokens is this?" is the question underneath every practical decision you will make about
an LLM application. Let's measure rather than guess.

### 8.1 Predict first

Below are three texts of almost exactly the same length in characters: an English ticket, a Bangla
ticket, and a few lines of Python.

**Before you run the cell, write down which one you think costs the most tokens per character.**
Genuinely write it down — the point of this exercise is destroyed if you read the answer first.

In [ ]:
samples = {
    "English": "Where is my order? I ordered the AeroSound Pro headphones last week and it has not moved.",
    "Bangla":  "আমার অর্ডার কোথায়? আমি গত সপ্তাহে হেডফোন অর্ডার করেছি কিন্তু এখনো কিছু আসেনি বলে জানাচ্ছি।",
    "Python":  "def triage(text: str) -> dict:\n    return {'category': 'order_status', 'urgency': 3}",
}

print(f"{'text':10} {'tokens':>7} {'chars':>7} {'chars/token':>12}")
print("-" * 39)
for label, text in samples.items():
    n = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    print(f"{label:10} {n:>7} {len(text):>7} {len(text) / n:>12.2f}")

**Most people guess Bangla, and most people are wrong.**

Bangla and English come out close to each other. **Code is the expensive one** — identifiers,
indentation, brackets and punctuation each fragment into their own tokens, so the same number of
characters costs far more.

The belief that non-Latin scripts cost several times more is not superstition; it was *emphatically
true* on GPT-2-era tokenizers, and the blog posts written then still rank first in search results
today. Modern tokenizers handle Bangla well. The measurement changed; the folklore didn't.

**The lesson that outlives these numbers:** you just contradicted a widely repeated claim in four
lines of code. Do that, rather than trusting an article older than the model you are using. Exercise
⭐ in `exercises.ipynb` makes you do it again on your own text.

### 8.2 Reading the meter on a real call

`count_tokens` is for planning *before* you send anything. For what a call actually consumed, read
`usage` on the response — and get into the habit of reading it, because it is the number that turns
into a bill.

Watch `total_thought_tokens` in particular. Modern models can do internal reasoning before they
answer, and **you pay for those tokens even though you never see them.** On `gemini-3.1-flash-lite`
it will usually read `0`, because a lite model does little or no thinking by default — that is a
large part of why it is cheap. Run the same cell against `gemini-2.5-pro` on a hard question and the
thinking count can exceed the visible answer several times over. A cost estimate built from the
words on screen would be wrong by that entire multiple.

In [ ]:
interaction = client.interactions.create(
    model=MODEL,
    input=f"In one sentence, what does this customer want?\n\n{ticket['body']}",
)

u = interaction.usage
print(interaction.output_text.strip())
print()
print(f"input tokens    : {u.total_input_tokens}")
print(f"output tokens   : {u.total_output_tokens}")
print(f"thinking tokens : {u.total_thought_tokens}")
print(f"total           : {u.total_tokens}")
print()

# ShopWise handles roughly 200 tickets a day. What does triage alone cost, in tokens?
per_day = u.total_tokens * 200
print(f"200 tickets/day  ≈ {per_day:,} tokens/day")
print(f"                 ≈ {per_day * 30:,} tokens/month")

**Notice what this notebook refuses to tell you: a price.** Rates change, tiers change, and any
number printed here would be wrong within a term. What you are learning is the *habit* — read
`usage` on every response, multiply by whatever today's published rate is, and know the number before
you scale anything to production. From session 4, LangSmith records this automatically for every
call you make for the rest of the course.

And before you scale that monthly figure in your head: most of the cost in a real assistant is not
triage. It is the conversation history from §7.7 and the retrieved documents from session 8, both of
which multiply the *input* side. Which is exactly why we measure.

### 8.3 Two things about the free tier, before you paste anything into a notebook

**Your free-tier prompts are used to train Google's models.** The pricing page lists "content used to
improve our products" as **Yes** for the free tier and **No** for paid. So never paste real customer
data, real names, card numbers, or anything belonging to an employer into a free-tier notebook. Every
ticket in `data/tickets.yml` is invented precisely so that this course needs nothing real. This is
not a formality — it is the first thing anyone will ask you about in a job interview on this topic.

**Rate limits are no longer published.** Google removed the requests-per-minute table from the docs;
it now says limits "can be viewed in Google AI Studio". So distrust any number in any blog post —
including any number in this notebook. Check your own:

- the docs page: https://ai.google.dev/gemini-api/docs/rate-limits
- your own dashboard: https://aistudio.google.com/rate-limit

We walk through both in class. If a cell starts returning `429 RESOURCE_EXHAUSTED`, that is a rate
limit and not a bug in your code: wait a minute and re-run.

## 9. The wider world: models, providers, and how to choose

Everything above used one model from one company. That was a teaching decision, not a
recommendation. This section is the map of the field, so you can hold an informed conversation about
it — in a design review, or an interview.

### 9.1 How we got here, briefly

The whole field runs on one architecture published in 2017. Everything since is scale, data and
tuning on top of it.

In [ ]:
mermaid("""
graph TD
    T["<b>2017</b> · 'Attention Is All You Need'<br/>Google publishes the Transformer<br/><i>everything below is built on this</i>"]
    G1["<b>2018–2020</b> · GPT-1 → GPT-3<br/>the only change that mattered was scale<br/>GPT-3: 175 billion parameters"]
    C["<b>Nov 2022</b> · ChatGPT<br/>the model wasn't new — the <b>chat interface</b> was<br/>fastest-adopted software product in history"]
    G4["<b>2023</b> · GPT-4 · LLaMA<br/>multimodal input arrives · Meta open-sources weights<br/>the closed/open split begins"]
    N["<b>2024–2026</b> · the current era<br/>million-token context · reasoning models · agents<br/>open weights close on the frontier"]

    T --> G1 --> C --> G4 --> N

    style T fill:#eaf7fc,stroke:#59c1e8
    style C fill:#e7f6ee,stroke:#2e9e5b
    style N fill:#2e9e5b,stroke:#14212e,color:#ffffff
""")

Two things in that timeline are worth more than the dates.

**ChatGPT was not a technical breakthrough.** The underlying model was already about two years old.
What changed in November 2022 was that someone put a *chat box* on it and made it free. The lesson
for you, as an applications engineer, is the one this entire course is built on: the value was in the
product wrapped around the model, not in the model.

**The pace is not slowing, and this is why we teach architecture rather than a model.** Every
capability you used in §7 tonight — reading a photo, searching the live web, returning strict JSON —
was a research demo or unavailable at consumer scale three years ago. The specific model IDs in this
notebook will be retired. The *shape* of the application you are learning to build will not.

### 9.2 Who makes the models

The field splits along one line that matters more than any benchmark: **can you download the
weights?**

| | **Closed weights** (API only) | **Open weights** (download and run) |
|---|---|---|
| Who | OpenAI (GPT) · Anthropic (Claude) · Google (Gemini) · xAI (Grok) | Meta (Llama) · Mistral · Alibaba (Qwen) · DeepSeek · Google (Gemma) |
| You get | an endpoint and a bill | files you can run on your own hardware |
| Best for | frontier capability, no ops burden | privacy, cost control, no vendor lock-in, fine-tuning |
| Trade-off | your data leaves your building; prices and models change under you | you own the GPUs, the latency, the uptime and the upgrades |

"Open weights" is worth stating precisely, because people use "open source" loosely here: you get the
trained parameters, usually under a licence with conditions. You very rarely get the training data or
the code that produced them. It is closer to a free binary than to open-source software.

**When it genuinely matters:** a Bangladeshi hospital or bank that cannot send patient or account
data to a US API does not have a "which model scores higher" problem. It has a "must run on our own
hardware" requirement, and that decides the question before benchmarks are opened.

### 9.3 Leaderboards — and how to read one without being fooled

New models arrive constantly and every one of them launches with a chart showing it winning.
Self-reported benchmarks are marketing. The most useful independent signal is a **community arena**:

**→ https://arena.ai/leaderboard**

How it works: you type a prompt, two anonymous models answer, you pick the better one, and only then
are you told which was which. Millions of those blind votes become an **Elo rating** — the same
system that ranks chess players. Nobody can train specially for it, because the questions are
whatever real people happened to ask.

Read it with three caveats:

1. **It measures human preference, not correctness.** A confident, well-formatted, slightly wrong
   answer beats a hedged correct one, because that is what people click.
2. **First place is usually noise.** The top several models are typically within the margin of error
   of each other. "Top five" is a real signal; "#1 this week" mostly is not.
3. **Use the category boards.** Overall rank is nearly useless for a specific job. There are separate
   boards for coding, maths, long context and vision, and they do not agree with each other.

Also worth bookmarking: **artificialanalysis.ai**, which plots quality against price and speed —
usually a better decision aid than rank alone, because for ShopWise the honest question is not "which
model is best" but "which is *good enough* at this task for the least money".

### 9.4 Hugging Face — the field's public library

**→ https://huggingface.co**

If a model has open weights, it is almost certainly on Hugging Face. As of 2026 the Hub holds well
over two million models and hundreds of thousands of datasets. Four things live there:

- **Models** — downloadable weights with a model card describing training, licence and limitations.
  Read the licence before you build a business on one.
- **Datasets** — public training and evaluation data. Where you would look to find, or publish, a
  Bangla-language dataset.
- **Spaces** — hosted demos. The fastest way to *try* a model before installing anything.
- **`transformers`** — the library that runs most of them locally, in a few lines of Python.

You will not download weights in this course; ShopWise runs on an API and the laptops in this room
are not GPU servers. But knowing where the open half of the field lives is part of being literate in
it, and the ⭐⭐⭐ exercise sends you there to look around.

### 9.5 Inference providers — the other half of the market

Downloading Llama does not mean you want to operate it. Running open weights well needs GPUs,
batching, autoscaling and someone awake at 3am. So a whole industry exists to run open models *for*
you, behind an API that mostly imitates OpenAI's:

| Provider | What it is known for |
|---|---|
| **Groq** (groq.com) | custom hardware; open models at remarkable speed |
| **Together AI** (together.ai) | broad catalogue of open models, plus fine-tuning |
| **Fireworks / Replicate** | fast hosted inference, image and video models too |
| **OpenRouter** (openrouter.ai) | one API key and one endpoint in front of **hundreds** of models, closed and open, with live prices |

OpenRouter deserves the extra sentence. It is a router, not a model maker: you change a string and
you are talking to a different company's model, with pricing visible per token. For comparing
candidates before committing, it saves a great deal of account-creation.

**Why any of this matters to what we're building:** it means the model is a *replaceable component*.
Session 4 introduces LangChain precisely so that swapping Gemini for Claude, or for a Llama running
at Groq, is a one-line change instead of a rewrite. You are not learning Gemini. You are learning to
build systems that happen to have a model in them.

### 9.6 What this course uses, and why

| Decision | Choice | Reason |
|---|---|---|
| Provider | Google Gemini | genuinely useful free tier — nobody needs a credit card to attend |
| Default model | `gemini-3.1-flash-lite` | cheapest and fast; good enough for almost everything we do |
| Stronger model | `gemini-3.6-flash` | when quality matters more than cost |
| Reasoning model | `gemini-2.5-pro` | hard problems, slower and dearer |
| Embeddings | `gemini-embedding-2` | session 8's retrieval |

**You will notice newer models in the listing below than the ones in that table** — `gemini-3.5-flash`
and `gemini-3.7-flash` both exist as you read this, and 3.7 is newer than the 3.6 we call our
"stronger" model. That is deliberate, and worth understanding, because it is the same call you will
make at work:

- **Newer is not automatically better *for a given task*.** Our default handles ticket triage,
  reading a photo and returning JSON perfectly well. Paying more per token for a task the cheap model
  already passes is a cost, not an upgrade.
- **A course needs stable ground.** Every output committed in this repo was executed against these
  IDs, so what you read matches what you get.
- **You should still try the newer ones.** Change `MODEL` in the setup cell, re-run §7, and see
  whether anything you actually care about improves. That experiment *is* the skill — and it is the
  honest version of the leaderboard question from §9.3.

Model IDs also get retired outright. So rather than trust any table, ask the API what exists today:

In [ ]:
available = sorted(m.name.removeprefix("models/") for m in client.models.list())

print(f"{len(available)} models visible to this key\n")
for name in available:
    if name.startswith("gemini-3"):
        print(" ", name)

print("\n… plus embedding, image, video, audio and open Gemma models.")
print("If a model ID in this notebook 404s in a future term, this cell is where you find its replacement.")

## 10. Setup lab

Everything above runs on an environment that works. Let's prove yours does.

The checker talks to nothing and spends no quota — it just tells you, line by line, whether your
machine is ready, and exactly what to do about anything that isn't.

From a terminal in `labs/`:

```bash
uv sync
cp .env.example .env      # then paste your key from aistudio.google.com/apikey
uv run python session-01/check_setup.py
```

Green all the way down means you are ready for session 2. Any `FAIL` line ends with the fix — do the
**first** one, run it again, and raise your hand if it still won't go green. A later failure is often
just a symptom of an earlier one.

The two that catch most people:

- **The notebook is running a different Python than `uv sync` installed into.** Pick the `.venv`
  kernel in Jupyter or VS Code. This is the cause of nearly every `ModuleNotFoundError` you will see
  this term.
- **The key was pasted with quotes around it.** `GEMINI_API_KEY=AIza...`, not
  `GEMINI_API_KEY="AIza..."`.

This is what lab time is for. A red checker on Saturday costs you the whole session.

## 11. What you now know, and what happens next

| Before today | After today |
|---|---|
| "AI" as one undifferentiated thing | AI ⊃ ML ⊃ deep learning ⊃ generative AI ⊃ LLMs, and why we work in the last one |
| An LLM writes text | it reads photos, searches the live web with sources, returns validated JSON, and streams |
| Python "which I sort of know" | a 22-part reference that names the session each part is needed in |
| Cost is per request | cost is per token, both directions, thinking included — and code costs more than prose |
| One model, one company | closed vs open weights, arena leaderboards, Hugging Face, inference providers |
| No environment | `uv`, `.env`, checker green |

### Exercises — before Saturday

`exercises.ipynb`, three of them, tagged ⭐ / ⭐⭐ / ⭐⭐⭐. Do them before session 2 — the ⭐⭐ one is
the scaffold session 2 builds on.

### Next session — Talking to the machines

You stop watching a demo and start writing the calls. Your own key, your own requests, and the three
roles — **system, user, assistant** — that every chat application on earth is built from. ShopWise
**v0.1** gets committed to your repo.

And you meet the problem that shapes the next eight sessions. Tonight the model looked capable.
Tomorrow we ask it about ShopWise's actual return policy, and it will answer — fluently, warmly,
with total confidence, and completely invented. It has never heard of ShopWise. Nothing in the
machinery distinguishes a fact it was told from a plausible sentence.

*"It has a brain, no hands, and no memory. Tomorrow we start fixing that."*